# Bayesian Non-parametric Causal Inference

In [1]:
import arviz as az
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from scipy.stats import norm
import pandas as pd
import geopandas as gpd
import statsmodels.formula.api as smf
import pymc as pm
import pymc_bart as pmb
import nutpie
import pytensor.tensor as pt
import os

In [2]:
%config InlineBackend.figure_format = 'retina'  # high resolution figures
az.style.use("arviz-darkgrid")
rng = np.random.default_rng(42)

# Estimating Treatment Effects

In this section we build a set of functions to pull out and extract a sample from our posterior distribution of propensity scores; use this propensity score to reweight the observed outcome variable across our treatment and control groups to re-calculate the average treatment effect (ATE). It reweights our data using one of three inverse probability weighting schemes and then plots three views (1) the raw propensity scores across groups (2) the raw outcome distribution and (3) the re-weighted outcome distribution.

First we define a bunch of helper functions for each weighting adjustment method we will explore:

In [3]:
# We need to do this process once for every iteration x chain (i.e., thousands of times), so we'll use statsmodel OLS rather than a complex Bayesian model like BART
def fit_outcome_model(X, predictors, y, model):

    X["y"] = y
    formula = 'y ~ ' + ' + '.join(predictors)+ ' + C(cbsa)'
    y_fit = smf.ols(formula=formula, data=X).fit()
    print(y_fit.summary())
    y_pred = y_fit.predict(X)
    return y_pred

def make_doubly_robust_adjustment(y_pred, y, ips):
    
    dr_estimate = y_pred + ips * (y - y_pred)

    return dr_estimate

def get_ate(y_pred, t, y, ips):
    
    dr_estimate = make_doubly_robust_adjustment(
        y_pred, y, ips
    )
    
    bandwidth = 0.025 * (np.max(t) - np.min(t))  # Example: 5% of range
    
    t_high, t_low = np.percentile(t, [75, 25])
    
    trt = np.mean(dr_estimate[(t >= t_high - bandwidth) & (t <= t_high + bandwidth)])
    ntrt = np.mean(dr_estimate[(t >= t_low - bandwidth) & (t <= t_low + bandwidth)])
    
    ate = np.exp(trt - ntrt)
    return [ate, trt, ntrt]

# Transportation

In [4]:
tran_par_file = '/work/hawkinslab/jfhawkin/SH-BART/data/vulcan/parquet/vulcan_x_epa_onroad_mn_2015_climate.parquet'
tran_df = gpd.read_parquet(tran_par_file)
tran_wgt_file = '/work/hawkinslab/jfhawkin/SH-BART/data/tran_weights.csv'
tran_wgt_df = pd.read_csv(tran_wgt_file)
# Create columns that assign decile numbers for each treatment variable
tran_df['treat_density'] = tran_df['d1a']
tran_df['treat_diversity'] = tran_df['d2b_e5mixa']

In [5]:
tran_wgt_df.shape

(215788, 5)

In [6]:
tran_df.shape

(215788, 382)

In [7]:
tran_df = pd.concat([tran_df,tran_wgt_df], axis=1)

tran_df = tran_df[(tran_df["dn_weighte"]>0) & (tran_df["totpop"]>0)]

# # Try removing extreme outliers
# lower_bound = tran_df.dn_weighte.quantile(0.01)
# upper_bound = tran_df.dn_weighte.quantile(0.99)

# tran_df = tran_df[(tran_df.dn_weighte > lower_bound) & (tran_df.dn_weighte < upper_bound)].dropna()

In [8]:
tran_df.shape

(215312, 387)

TEMP!!!!

In [9]:
def weighted_avg(values, weights):
    return (values * weights.sum(axis=1)).sum() / weights.sum(axis=1).sum()

grp_col = "cbsa"
val_col = "d4a"
agg_val_col = "d4a_cbsa"

tran_df_cbsa = tran_df.drop_duplicates(subset=grp_col).copy()

wgt_cols = ["totpop"]
tran_df_cbsa.loc[:, val_col] = tran_df.groupby(grp_col).apply(lambda x: weighted_avg(x[val_col].replace(-99999,1), x.loc[:, wgt_cols]), include_groups=False # Use 1 not 0 for ln()
    ).values

d4a_dict = dict(zip(tran_df_cbsa[grp_col], tran_df_cbsa[val_col]))
tran_df[agg_val_col] = tran_df[grp_col].map(d4a_dict) # Need to use a merging method

In [10]:
predictors = ["totpop_cbsa",
            "d1a_cbsa"]
            #  "d2b_e5mix_cbsa",
            #   "d3a_cbsa",
            #  "d4a_cbsa", 
            #  "d5ar_cbsa"]

X = tran_df.copy()
    
X["d1aa_cbsa"] = tran_df["totpop_cbsa"]/tran_df["ac_unpr_cbsa"] # true cbsa density

y = tran_df["dn_weighte"] / tran_df["totpop"].replace(0, np.nan) # make per capita

X.loc[:,predictors] = np.log(X.loc[:,predictors])
y = np.log(y)
predictors+=["ps"]

/tmp/ipykernel_3581829/1031059109.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[12.94226974 12.94226974 12.94226974 ... 11.88683798 12.03706802
 12.03706802]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X.loc[:,predictors] = np.log(X.loc[:,predictors])


In [11]:
y.isna().sum()

0

## Density

In [12]:
X["ps"] = X["dens"]
ips = 1/X["ps"]
t_density = tran_df["treat_density"]

out = fit_outcome_model(X, predictors, y, "tran_dens")

ate = get_ate(out, t_density, y, ips)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.174
Model:                            OLS   Adj. R-squared:                  0.170
Method:                 Least Squares   F-statistic:                     46.96
Date:                Thu, 17 Apr 2025   Prob (F-statistic):               0.00
Time:                        20:21:54   Log-Likelihood:            -2.9251e+05
No. Observations:              215312   AIC:                         5.869e+05
Df Residuals:                  214349   BIC:                         5.968e+05
Df Model:                         962                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            3.6828      0.309  

In [13]:
y.describe()

count    215312.000000
mean         -0.104331
std           1.035835
min         -10.732578
25%          -0.794157
50%          -0.139076
75%           0.544252
max           9.231250
dtype: float64

In [14]:
out.describe()

count    215312.000000
mean         -0.104331
std           0.432161
min          -1.413057
25%          -0.382332
50%          -0.098767
75%           0.223561
max           1.610082
dtype: float64

In [15]:
X.ps.describe()

count    215312.000000
mean          0.957752
std           0.506791
min           0.110582
25%           0.557222
50%           1.009309
75%           1.353050
max           1.716024
Name: ps, dtype: float64

In [16]:
ips.describe()

count    215312.000000
mean          2.084830
std           2.545820
min           0.582742
25%           0.739071
50%           0.990777
75%           1.794616
max           9.043046
Name: ps, dtype: float64

In [17]:
ate

[0.9863965594710917, -0.2310168812273994, -0.21732006612407845]

In [18]:
np.exp(y - out).describe()

count    215312.000000
mean          2.301885
std          72.078048
min           0.000015
25%           0.541857
50%           0.928189
75%           1.713339
max       23173.584231
dtype: float64

In [19]:
y[X.y==X.y.max()]

151352    9.23125
dtype: float64

In [20]:
out[X.y==X.y.max()]

151352   -0.819518
dtype: float64